# Maratończycy: `groupby`, `pivot_table`, wykresy i interpretacja

Przykładowy notebook do zajęć.

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_PATH = Path('/mnt/data') / 'marathon-data.csv'
df = pd.read_csv(DATA_PATH)
df.head()


In [ ]:

df.info()


## 1. Przygotowanie danych

In [ ]:

for col in ['split', 'final']:
    df[col + '_td'] = pd.to_timedelta(df[col])
    df[col + '_min'] = df[col + '_td'].dt.total_seconds() / 60

df['slowdown_min'] = df['final_min'] - 2 * df['split_min']

age_bins = [15,19,24,29,34,39,44,49,54,59,64,69,100]
age_labels = ['16-19','20-24','25-29','30-34','35-39','40-44','45-49','50-54','55-59','60-64','65-69','70+']
df['age_group'] = pd.cut(df['age'], bins=age_bins, labels=age_labels)

df.head()


## 2. `groupby`: która grupa wiekowa jest najszybsza?

In [ ]:

age_summary = (
    df.groupby('age_group', observed=False)
      .agg(
          n=('final_min', 'size'),
          mean_final=('final_min', 'mean'),
          median_final=('final_min', 'median'),
          mean_split=('split_min', 'mean'),
          mean_slowdown=('slowdown_min', 'mean')
      )
      .reset_index()
)

age_summary.round(2)


In [ ]:

age_summary.sort_values('mean_final').head(5).round(2)


In [ ]:

age_summary.sort_values('median_final').head(5).round(2)


## 3. `pivot_table`: wiek × płeć

In [ ]:

pivot_mean = pd.pivot_table(
    df,
    index='age_group',
    columns='gender',
    values='final_min',
    aggfunc='mean',
    observed=False
)

pivot_median = pd.pivot_table(
    df,
    index='age_group',
    columns='gender',
    values='final_min',
    aggfunc='median',
    observed=False
)

pivot_count = pd.pivot_table(
    df,
    index='age_group',
    columns='gender',
    values='final_min',
    aggfunc='size',
    observed=False
)

pivot_mean.round(2), pivot_median.round(2), pivot_count


## 4. Wykresy

In [ ]:

plt.figure(figsize=(10,4))
plt.plot(age_summary['age_group'].astype(str), age_summary['mean_final'], marker='o', label='mean_final')
plt.plot(age_summary['age_group'].astype(str), age_summary['median_final'], marker='o', label='median_final')
plt.xticks(rotation=35)
plt.ylabel('minuty')
plt.title('Czas końcowy vs grupa wiekowa')
plt.legend()
plt.show()


In [ ]:

pivot_mean.round(2).plot(kind='bar', figsize=(10,4))
plt.ylabel('średni czas końcowy [min]')
plt.title('Średni wynik końcowy: grupa wiekowa × płeć')
plt.show()


## 5. Pytania dla studentów

1. Która grupa wiekowa jest najszybsza po średniej, a która po medianie?
2. Czy ranking zmienia się po rozbiciu na płeć?
3. Które grupy najbardziej zwalniają po połowie dystansu?
4. Czy każda interpretacja jest równie wiarygodna przy małych liczebnościach grup?

In [ ]:

# TODO dla studentów:
# 1. Zrób pivot_table dla slowdown_min po age_group i gender
# 2. Posortuj age_summary po mean_slowdown
# 3. Narysuj własny wykres pokazujący różnice między grupami
